# Chapter 13 — Serving and Systems Tradeoffs: Exercises

Eight exercises covering bandwidth-bound decode, KV cache budgets,
batching strategies, PagedAttention, speculative decoding,
disaggregation, cost optimisation, and queue stability.

In [ ]:
# === Setup ===
import numpy as np
np.random.seed(42)

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

from math import factorial, ceil

def print_table(headers, rows, col_width=16):
    """Print a formatted table."""
    hdr = " | ".join(h.center(col_width) for h in headers)
    print(hdr)
    print("-" * len(hdr))
    for row in rows:
        print(" | ".join(str(v).center(col_width) for v in row))

def fmt_bytes(val):
    """Format bytes with appropriate unit."""
    for suffix, div in [("TB", 1e12), ("GB", 1e9), ("MB", 1e6), ("KB", 1e3)]:
        if abs(val) >= div:
            return f"{val/div:.2f} {suffix}"
    return f"{val:.0f} B"

print("Setup complete ✓")

## Exercise 1: Latency Modelling

**Given:** LLaMA-3 8B (8 × 10⁹ params) in BF16 on a single H100:
- HBM bandwidth: 3.35 TB/s
- Peak BF16 FLOP/s: 989 TFLOP/s

**Tasks:**
1. Compute the bandwidth-bound TPOT at batch sizes B = 1, 8, 32, 128.
2. Compute the critical batch size B* where the system transitions from
   bandwidth-bound to compute-bound.
3. At what batch size does per-user TPOT start increasing?

In [ ]:
# === Exercise 1: Scaffold ===

P = 8e9           # parameters
b = 2             # bytes per param (BF16)
BW = 3.35e12      # HBM bandwidth (bytes/s)
PEAK = 989e12     # peak BF16 FLOP/s

# TODO 1: For each B in [1, 8, 32, 128], compute:
#   - time per decode step (seconds) = max(P*b/BW, 2*P*B/PEAK)
#   - total throughput (tokens/s) = B / time_per_step
#   - per-user TPOT (ms) = time_per_step / B * 1000

# TODO 2: Compute B* = PEAK * b / (2 * BW)

# TODO 3: TPOT starts increasing when B > B*

pass

In [ ]:
# === Exercise 1: Solution ===

P = 8e9
b = 2
BW = 3.35e12
PEAK = 989e12

print("=" * 70)
print("EXERCISE 1: LATENCY MODELLING — LLaMA-3 8B, BF16, H100")
print("=" * 70)

# Part 1: TPOT at various batch sizes
print("\nPart 1: Decode metrics at various batch sizes")
print()

headers = ["Batch B", "t_step (ms)", "Throughput", "TPOT (ms)", "Regime"]
rows = []

for B in [1, 8, 32, 128]:
    t_bw = P * b / BW                    # bandwidth-bound time per step
    t_compute = 2 * P * B / PEAK         # compute-bound time per step
    t_step = max(t_bw, t_compute)        # actual time per step
    throughput = B / t_step              # total tokens/s
    tpot = t_step / B * 1000            # per-user ms
    regime = "BW-bound" if t_bw >= t_compute else "Compute-bound"

    rows.append([
        str(B),
        f"{t_step*1000:.3f}",
        f"{throughput:,.0f} tok/s",
        f"{tpot:.3f}",
        regime
    ])

print_table(headers, rows, col_width=16)

# Part 2: Critical batch size
B_star = PEAK * b / (2 * BW)
print(f"\nPart 2: Critical batch size B* = PEAK × b / (2 × BW)")
print(f"  B* = {PEAK:.0e} × {b} / (2 × {BW:.2e})")
print(f"  B* = {B_star:.1f}")
print(f"  At B > {B_star:.0f}, the system becomes compute-bound.")

# Part 3: When TPOT starts increasing
print(f"\nPart 3: TPOT starts increasing when B > B* = {B_star:.0f}")
print(f"  In the BW-bound regime (B ≤ {B_star:.0f}), TPOT is constant:")
tpot_bw = P * b / BW * 1000
print(f"    TPOT = P × b / BW = {tpot_bw:.3f} ms")
print(f"  Above B*, TPOT grows linearly with B because the GPU")
print(f"  can no longer process all tokens in one weight-read pass.")

## Exercise 2: KV Cache Budget

**Given:** LLaMA-3 70B on 2× H100-80GB with TP=2:
- 80 layers, GQA with 8 KV heads, d_head = 128
- BF16 KV cache (2 bytes per element)

**Tasks:**
1. Compute KV cache bytes per token per layer.
2. Compute KV cache bytes per token (all layers) per GPU.
3. Compute available KV memory per GPU (after model weights).
4. Find maximum concurrent tokens and concurrent 2048-token requests.

In [ ]:
# === Exercise 2: Scaffold ===

n_layers = 80
n_kv_heads = 8      # GQA
d_head = 128
b_kv = 2            # BF16
P_70B = 70e9
b_model = 2         # BF16 weights
gpu_mem = 80e9      # 80 GB per GPU
tp = 2
overhead = 2e9      # ~2 GB for activations + framework

# TODO 1: kv_per_tok_per_layer = 2 * n_kv_heads * d_head * b_kv
#         (factor 2 for K and V)

# TODO 2: kv_per_tok_total = n_layers * kv_per_tok_per_layer
#         kv_per_tok_per_gpu = kv_per_tok_total / tp

# TODO 3: model_mem_per_gpu = P_70B * b_model / tp
#         avail = gpu_mem - model_mem_per_gpu - overhead

# TODO 4: max_tokens = avail / kv_per_tok_per_gpu
#         max_requests_2048 = max_tokens / 2048

pass

In [ ]:
# === Exercise 2: Solution ===

n_layers = 80
n_kv_heads = 8
d_head = 128
b_kv = 2
P_70B = 70e9
b_model = 2
gpu_mem = 80e9
tp = 2
overhead = 2e9

print("=" * 70)
print("EXERCISE 2: KV CACHE BUDGET — LLaMA-3 70B, BF16, 2×H100, TP=2")
print("=" * 70)

# Part 1: KV per token per layer
kv_per_tok_per_layer = 2 * n_kv_heads * d_head * b_kv
print(f"\nPart 1: KV per token per layer")
print(f"  = 2 × {n_kv_heads} × {d_head} × {b_kv}")
print(f"  = {kv_per_tok_per_layer:,} bytes = {fmt_bytes(kv_per_tok_per_layer)}")

# Part 2: KV per token (all layers) per GPU
kv_per_tok_total = n_layers * kv_per_tok_per_layer
kv_per_tok_per_gpu = kv_per_tok_total / tp
print(f"\nPart 2: KV per token (all layers, per GPU)")
print(f"  Total: {n_layers} × {kv_per_tok_per_layer:,} = {kv_per_tok_total:,} bytes = {fmt_bytes(kv_per_tok_total)}")
print(f"  Per GPU (TP={tp}): {fmt_bytes(kv_per_tok_per_gpu)}")

# Part 3: Available KV memory
model_mem_per_gpu = P_70B * b_model / tp
avail = gpu_mem - model_mem_per_gpu - overhead
print(f"\nPart 3: Available KV memory per GPU")
print(f"  GPU memory:      {fmt_bytes(gpu_mem)}")
print(f"  Model weights:   {fmt_bytes(model_mem_per_gpu)}")
print(f"  Overhead:        {fmt_bytes(overhead)}")
print(f"  Available:       {fmt_bytes(avail)}")

# Part 4: Max concurrent tokens and requests
max_tokens = avail / kv_per_tok_per_gpu
print(f"\nPart 4: Maximum concurrency")
print(f"  Max tokens:      {max_tokens:,.0f}")
for ctx in [512, 1024, 2048, 4096, 8192]:
    max_req = max_tokens / ctx
    print(f"  @ {ctx:>5} tokens:  {max_req:.0f} concurrent requests")

## Exercise 3: Continuous vs. Static Batching Throughput

**Given:** 10 requests with output lengths:
[50, 75, 100, 125, 150, 175, 200, 250, 300, 400]

Batch size = 10. TPOT = 10 ms.

**Tasks:**
1. Compute total time under static batching.
2. Compute total time under continuous batching
   (assume new requests arrive instantly as slots free).
3. Calculate the throughput improvement.

In [ ]:
# === Exercise 3: Scaffold ===

output_lengths = np.array([50, 75, 100, 125, 150, 175, 200, 250, 300, 400])
BATCH_SIZE = 10
TPOT = 10  # ms

# TODO 1: Static batching time
#   All requests padded to max length (400)
#   static_time_ms = max(output_lengths) * TPOT
#   total_tokens = sum(output_lengths)

# TODO 2: Continuous batching
#   Simulate slot-by-slot: each step takes TPOT ms,
#   all 10 slots generate 1 token per step.
#   When a slot finishes, it has no replacement (only 10 requests total).

# TODO 3: Speedup = static_time / continuous_time

pass

In [ ]:
# === Exercise 3: Solution ===

output_lengths = np.array([50, 75, 100, 125, 150, 175, 200, 250, 300, 400])
BATCH_SIZE = 10
TPOT = 10  # ms

print("=" * 70)
print("EXERCISE 3: CONTINUOUS vs STATIC BATCHING")
print("=" * 70)

total_tokens = output_lengths.sum()
print(f"\nOutput lengths: {list(output_lengths)}")
print(f"Total tokens: {total_tokens}")
print(f"TPOT: {TPOT} ms")

# Part 1: Static batching
# All requests must wait until the longest one finishes
s_max = output_lengths.max()
static_time_ms = s_max * TPOT  # all batch slots run for max length
static_compute_tokens = BATCH_SIZE * s_max  # total token-slots used
static_throughput = total_tokens / (static_time_ms / 1000)  # useful tok/s

print(f"\nPart 1: Static Batching")
print(f"  Max output length: {s_max}")
print(f"  Batch time: {s_max} × {TPOT} = {static_time_ms:,} ms = {static_time_ms/1000:.1f} s")
print(f"  Token-slots used: {static_compute_tokens:,} (wasted: {static_compute_tokens - total_tokens:,})")
print(f"  Useful throughput: {static_throughput:,.0f} tok/s")

# Part 2: Continuous batching
# Simulate step by step: only 10 requests, no replenishment
remaining = list(output_lengths)
continuous_steps = 0
useful_tokens = 0

while any(r > 0 for r in remaining):
    continuous_steps += 1
    for i in range(len(remaining)):
        if remaining[i] > 0:
            remaining[i] -= 1
            useful_tokens += 1

continuous_time_ms = continuous_steps * TPOT
continuous_throughput = useful_tokens / (continuous_time_ms / 1000)

print(f"\nPart 2: Continuous Batching (no new arrivals)")
print(f"  Total steps: {continuous_steps}")
print(f"  Total time: {continuous_steps} × {TPOT} = {continuous_time_ms:,} ms = {continuous_time_ms/1000:.1f} s")
print(f"  Useful tokens: {useful_tokens:,}")
print(f"  Useful throughput: {continuous_throughput:,.0f} tok/s")

# Note: With no new arrivals, continuous batching still takes max_len steps,
# but active slots decrease over time (fewer compute wasted).
# The real benefit of continuous batching is with a *stream* of new arrivals.
# Let's also compute with infinite arrivals:

print(f"\n  Note: With finite requests and no arrivals, continuous and static")
print(f"  both finish in s_max = {s_max} steps.")
print(f"  The benefit: continuous batching frees slots early.")
print(f"  With a stream of arrivals, freed slots serve new requests.")

# Part 3: Speedup with infinite stream
# Static: processes N tokens in ceil(N/B) batches × s_max
# Continuous: processes N tokens with near-100% slot utilisation
mean_len = output_lengths.mean()
theoretical_speedup = s_max / mean_len

print(f"\nPart 3: Throughput improvement (with arrival stream)")
print(f"  Static throughput:     {BATCH_SIZE * 1000 / (s_max * TPOT):,.0f} tok/s/slot")
print(f"  Continuous throughput: {1000 / (mean_len * TPOT) * BATCH_SIZE:,.0f} tok/s/slot (ideal)")
print(f"  Speedup ≈ s_max / E[s] = {s_max} / {mean_len:.0f} = {theoretical_speedup:.2f}×")

## Exercise 4: PagedAttention Fragmentation

**Given:** 16 requests with sequence lengths drawn from Uniform(100, 2000).
- Block size = 16 tokens
- KV bytes per token = 320 KB (all layers)
- s_max = 4096

**Tasks:**
1. Compute expected naive waste (pre-allocated max length) in GB.
2. Compute expected paged waste (only last-block fragmentation) in GB.
3. How many additional requests can you serve with the saved memory?

In [ ]:
# === Exercise 4: Scaffold ===

np.random.seed(42)
N_REQ = 16
S_MAX = 4096
BLOCK_SIZE = 16
KV_PER_TOK = 320 * 1024  # 320 KB in bytes

seq_lengths = np.random.randint(100, 2001, size=N_REQ)

# TODO 1: Naive waste
#   Each request allocated S_MAX tokens
#   waste = sum(S_MAX - s_i) * KV_PER_TOK

# TODO 2: Paged waste
#   Each request uses ceil(s_i / BLOCK_SIZE) blocks
#   Allocated = sum(ceil(s_i / BS) * BS * KV_PER_TOK)
#   Used = sum(s_i * KV_PER_TOK)
#   Waste = Allocated - Used

# TODO 3: Saved memory / avg_kv_per_request = additional requests

pass

In [ ]:
# === Exercise 4: Solution ===

np.random.seed(42)
N_REQ = 16
S_MAX = 4096
BLOCK_SIZE = 16
KV_PER_TOK = 320 * 1024  # bytes

seq_lengths = np.random.randint(100, 2001, size=N_REQ)

print("=" * 70)
print("EXERCISE 4: PAGEDATTENTION FRAGMENTATION")
print("=" * 70)
print(f"\nSequence lengths: {list(seq_lengths)}")
print(f"Mean: {seq_lengths.mean():.0f}, Min: {seq_lengths.min()}, Max: {seq_lengths.max()}")
print(f"Block size: {BLOCK_SIZE}, KV/token: {fmt_bytes(KV_PER_TOK)}")

# Part 1: Naive waste
naive_allocated = N_REQ * S_MAX * KV_PER_TOK
naive_used = seq_lengths.sum() * KV_PER_TOK
naive_waste = naive_allocated - naive_used

print(f"\nPart 1: Naive Allocation")
print(f"  Allocated: {N_REQ} × {S_MAX} × {fmt_bytes(KV_PER_TOK)} = {fmt_bytes(naive_allocated)}")
print(f"  Used:      {seq_lengths.sum():,} × {fmt_bytes(KV_PER_TOK)} = {fmt_bytes(naive_used)}")
print(f"  Waste:     {fmt_bytes(naive_waste)}")
print(f"  Waste %:   {naive_waste/naive_allocated*100:.1f}%")

# Part 2: Paged waste
blocks_per_req = np.ceil(seq_lengths / BLOCK_SIZE).astype(int)
paged_allocated_tokens = (blocks_per_req * BLOCK_SIZE).sum()
paged_allocated = paged_allocated_tokens * KV_PER_TOK
paged_waste = paged_allocated - naive_used

print(f"\nPart 2: Paged Allocation (block_size={BLOCK_SIZE})")
print(f"  Total blocks: {blocks_per_req.sum()}")
print(f"  Allocated tokens: {paged_allocated_tokens:,}")
print(f"  Allocated: {fmt_bytes(paged_allocated)}")
print(f"  Waste:     {fmt_bytes(paged_waste)}")
print(f"  Waste %:   {paged_waste/paged_allocated*100:.1f}%")
print(f"  Reduction: {naive_waste/paged_waste:.0f}× less waste")

# Part 3: Additional requests
saved = naive_waste - paged_waste
avg_kv_per_req = seq_lengths.mean() * KV_PER_TOK
additional = saved / avg_kv_per_req

print(f"\nPart 3: Memory Savings")
print(f"  Saved:      {fmt_bytes(saved)}")
print(f"  Avg KV/req: {fmt_bytes(avg_kv_per_req)}")
print(f"  Additional requests: {additional:.0f} (at mean seq length {seq_lengths.mean():.0f})")

## Exercise 5: Speculative Decoding

**Given:**
- Target model: 70B (BF16), draft model: 8B (BF16)
- Draft length K = 4
- Mean acceptance rate α = 0.75
- Draft model is 10× faster than target per token (γ = 0.1)

**Tasks:**
1. Compute expected tokens per speculative cycle.
2. Compute speed-up over standard autoregressive decoding.
3. What value of α makes speculation break even (S = 1)?

In [ ]:
# === Exercise 5: Scaffold ===

alpha = 0.75
K = 4
gamma = 0.1  # t_draft / t_target

# TODO 1: E[accepted] = (1 - α^(K+1)) / (1 - α) - 1
#         tokens_per_cycle = E[accepted] + 1

# TODO 2: t_cycle = K * γ + 1 (normalised by t_AR)
#         speedup = tokens_per_cycle / t_cycle

# TODO 3: Find α where speedup = 1
#         n_cycle / (K*γ + 1) = 1
#         → (1 - α^(K+1))/(1-α) = K*γ + 1
#         Binary search or analytical solve

pass

In [ ]:
# === Exercise 5: Solution ===

alpha = 0.75
K = 4
gamma = 0.1

print("=" * 70)
print("EXERCISE 5: SPECULATIVE DECODING")
print("=" * 70)
print(f"\nα = {alpha}, K = {K}, γ = {gamma}")

# Part 1: Expected tokens per cycle
E_accepted = (1 - alpha**(K+1)) / (1 - alpha) - 1
n_cycle = E_accepted + 1

print(f"\nPart 1: Expected tokens per cycle")
print(f"  E[accepted] = (1 - {alpha}^{K+1}) / (1 - {alpha}) - 1")
print(f"              = (1 - {alpha**(K+1):.4f}) / {1-alpha} - 1")
print(f"              = {(1-alpha**(K+1))/(1-alpha):.4f} - 1")
print(f"              = {E_accepted:.4f}")
print(f"  Tokens/cycle = E[accepted] + 1 = {n_cycle:.4f}")

# Part 2: Speedup
t_cycle = K * gamma + 1
speedup = n_cycle / t_cycle

print(f"\nPart 2: Speedup")
print(f"  t_cycle (normalised) = K × γ + 1 = {K} × {gamma} + 1 = {t_cycle}")
print(f"  Speedup = {n_cycle:.4f} / {t_cycle} = {speedup:.4f}×")
print(f"  → {speedup:.2f}× faster than standard autoregressive decoding")

# Part 3: Break-even α
# At S=1: n_cycle = t_cycle → (1-α^(K+1))/(1-α) = K*γ+1
# Binary search
target = K * gamma + 1  # = 1.4
lo, hi = 0.001, 0.999
for _ in range(100):
    mid = (lo + hi) / 2
    lhs = (1 - mid**(K+1)) / (1 - mid)
    if lhs > target:
        hi = mid
    else:
        lo = mid
alpha_be = mid

print(f"\nPart 3: Break-even acceptance rate")
print(f"  Need: (1 - α^{K+1}) / (1 - α) = {target}")
print(f"  Break-even α = {alpha_be:.4f}")
print(f"  Below α = {alpha_be:.4f}, speculative decoding is SLOWER.")

# Verify
E_be = (1 - alpha_be**(K+1)) / (1 - alpha_be) - 1
S_be = (E_be + 1) / t_cycle
print(f"  Verification: S({alpha_be:.4f}) = {S_be:.4f} ≈ 1.0 ✓")

## Exercise 6: Prefill-Decode Disaggregation

**Given:**
- Avg prompt: 500 tokens, avg output: 200 tokens
- Prefill throughput: 50,000 input tok/s per GPU
- Decode throughput: 5,000 output tok/s per GPU
- Traffic: 100 req/s
- 20 total GPUs available

**Tasks:**
1. Compute GPU-seconds per request for prefill and decode separately.
2. Find optimal prefill:decode GPU ratio.
3. For 20 total GPUs, determine prefill vs. decode allocation.

In [ ]:
# === Exercise 6: Scaffold ===

avg_prompt = 500      # tokens
avg_output = 200      # tokens
prefill_tps = 50000   # input tok/s per GPU
decode_tps = 5000     # output tok/s per GPU
traffic = 100         # req/s
total_gpus = 20

# TODO 1: GPU-seconds per request
#   prefill_gpu_s = avg_prompt / prefill_tps
#   decode_gpu_s = avg_output / decode_tps

# TODO 2: Total GPU-seconds/s needed for each phase
#   prefill_demand = traffic * prefill_gpu_s
#   decode_demand = traffic * decode_gpu_s
#   ratio = prefill_demand / decode_demand

# TODO 3: Allocate total_gpus proportionally

pass

In [ ]:
# === Exercise 6: Solution ===

avg_prompt = 500
avg_output = 200
prefill_tps = 50000
decode_tps = 5000
traffic = 100
total_gpus = 20

print("=" * 70)
print("EXERCISE 6: PREFILL-DECODE DISAGGREGATION")
print("=" * 70)

# Part 1: GPU-seconds per request
prefill_gpu_s = avg_prompt / prefill_tps
decode_gpu_s = avg_output / decode_tps

print(f"\nPart 1: GPU-seconds per request")
print(f"  Prefill: {avg_prompt} / {prefill_tps:,} = {prefill_gpu_s:.4f} GPU-s")
print(f"  Decode:  {avg_output} / {decode_tps:,} = {decode_gpu_s:.4f} GPU-s")
print(f"  Decode is {decode_gpu_s/prefill_gpu_s:.0f}× more GPU-intensive per request")

# Part 2: Total demand and optimal ratio
prefill_demand = traffic * prefill_gpu_s  # GPU-s/s = GPUs needed
decode_demand = traffic * decode_gpu_s
total_demand = prefill_demand + decode_demand
ratio = prefill_demand / decode_demand

print(f"\nPart 2: GPU demand at {traffic} req/s")
print(f"  Prefill GPUs needed: {traffic} × {prefill_gpu_s:.4f} = {prefill_demand:.1f}")
print(f"  Decode GPUs needed:  {traffic} × {decode_gpu_s:.4f} = {decode_demand:.1f}")
print(f"  Total:               {total_demand:.1f} GPUs")
print(f"  Prefill:Decode ratio = {ratio:.2f} : 1")

# Part 3: Allocation of 20 GPUs
prefill_frac = prefill_demand / total_demand
decode_frac = decode_demand / total_demand
n_prefill = max(1, round(total_gpus * prefill_frac))
n_decode = total_gpus - n_prefill

# Check utilisation
prefill_util = prefill_demand / n_prefill
decode_util = decode_demand / n_decode

print(f"\nPart 3: Allocating {total_gpus} GPUs")
print(f"  Prefill frac: {prefill_frac:.2%} → {n_prefill} GPUs")
print(f"  Decode frac:  {decode_frac:.2%} → {n_decode} GPUs")
print(f"  Prefill utilisation: {prefill_util:.1%}")
print(f"  Decode utilisation:  {decode_util:.1%}")

# Also try nearby allocations
print(f"\n  Sensitivity analysis:")
headers = ["Prefill GPUs", "Decode GPUs", "Prefill Util", "Decode Util", "Balanced?"]
rows = []
for np_ in range(1, total_gpus):
    nd_ = total_gpus - np_
    pu = prefill_demand / np_
    du = decode_demand / nd_
    balanced = "✓" if abs(pu - du) < 0.1 else ""
    if 1 <= np_ <= 5:  # only show reasonable range
        rows.append([str(np_), str(nd_), f"{pu:.1%}", f"{du:.1%}", balanced])
print_table(headers, rows, col_width=14)

## Exercise 7: Cost Optimisation

**Given:** LLaMA-3 70B on 2× H100 at \$3.00/GPU/hr.
- BF16 throughput: 2,500 tok/s
- INT4 throughput: 5,500 tok/s
- Traffic: 1 billion output tokens/day

**Tasks:**
1. Compute CPM for BF16 and INT4.
2. Compute daily and monthly cost savings of INT4 vs BF16.
3. At what quality-loss threshold (fraction of tokens needing
   regeneration) does INT4 stop being worth it?

In [ ]:
# === Exercise 7: Scaffold ===

gpu_cost_hr = 3.00
n_gpus = 2
bf16_tps = 2500
int4_tps = 5500
daily_tokens = 1e9

# TODO 1: CPM = (gpu_cost_hr * n_gpus * 1e6) / (throughput * 3600)

# TODO 2: daily_cost = (daily_tokens / 1e6) * CPM
#         monthly_cost = daily_cost * 30
#         savings = bf16_cost - int4_cost

# TODO 3: If fraction f of INT4 tokens need regeneration,
#         effective INT4 tokens = daily_tokens * (1 + f)
#         Break-even when INT4 effective cost = BF16 cost

pass

In [ ]:
# === Exercise 7: Solution ===

gpu_cost_hr = 3.00
n_gpus = 2
bf16_tps = 2500
int4_tps = 5500
daily_tokens = 1e9

print("=" * 70)
print("EXERCISE 7: COST OPTIMISATION — BF16 vs INT4")
print("=" * 70)

# Part 1: CPM
total_cost_hr = gpu_cost_hr * n_gpus
cpm_bf16 = total_cost_hr * 1e6 / (bf16_tps * 3600)
cpm_int4 = total_cost_hr * 1e6 / (int4_tps * 3600)

print(f"\nPart 1: Cost Per Million Tokens (CPM)")
print(f"  Total GPU cost: ${total_cost_hr:.2f}/hr")
print(f"  BF16: CPM = ${total_cost_hr:.2f} × 10⁶ / ({bf16_tps:,} × 3600) = ${cpm_bf16:.4f}")
print(f"  INT4: CPM = ${total_cost_hr:.2f} × 10⁶ / ({int4_tps:,} × 3600) = ${cpm_int4:.4f}")
print(f"  INT4 is {cpm_bf16/cpm_int4:.2f}× cheaper per token")

# Part 2: Daily/monthly costs and savings
daily_bf16 = (daily_tokens / 1e6) * cpm_bf16
daily_int4 = (daily_tokens / 1e6) * cpm_int4
daily_savings = daily_bf16 - daily_int4
monthly_savings = daily_savings * 30

print(f"\nPart 2: Cost for {daily_tokens/1e9:.0f}B tokens/day")
print(f"  BF16 daily:   ${daily_bf16:>10,.2f}")
print(f"  INT4 daily:   ${daily_int4:>10,.2f}")
print(f"  Daily savings: ${daily_savings:>10,.2f}")
print(f"  Monthly savings: ${monthly_savings:>8,.2f}")

# Part 3: Quality-loss break-even
# If f = fraction needing regen, effective tokens = daily_tokens * (1 + f)
# INT4 cost with regen = (daily_tokens * (1 + f) / 1e6) * cpm_int4
# Break even: (1 + f) * cpm_int4 = cpm_bf16
f_breakeven = cpm_bf16 / cpm_int4 - 1

print(f"\nPart 3: Quality-loss break-even")
print(f"  (1 + f) × CPM_int4 = CPM_bf16")
print(f"  f = CPM_bf16/CPM_int4 - 1 = {cpm_bf16/cpm_int4:.4f} - 1 = {f_breakeven:.4f}")
print(f"  Break-even: {f_breakeven:.2%} of tokens need regeneration")
print(f"  If quality loss causes > {f_breakeven:.1%} regen rate, INT4 is NOT worth it.")
print(f"  Typical INT4 regen rate is < 1%, so INT4 is almost always beneficial.")

## Exercise 8: Queue Stability

**Given:** Poisson arrivals at λ = 20 req/s.
Each request takes on average 2 seconds to serve (μ = 0.5 req/s per GPU).

**Tasks:**
1. What is the minimum number of GPUs for a stable queue?
2. For the queue to have ρ < 0.7, how many GPUs are needed?
3. With the number from (2), compute expected wait time Wq using M/M/c.

In [ ]:
# === Exercise 8: Scaffold ===

lam = 20.0    # req/s
mu = 0.5      # req/s per GPU (avg service time = 2s)

# TODO 1: For stability: ρ = λ / (c × μ) < 1
#         c_min = ceil(λ / μ)

# TODO 2: For ρ < 0.7:
#         c = ceil(λ / (μ × 0.7))

# TODO 3: Use M/M/c formula to compute Wq
#   P0 = [Σ (c*ρ)^k/k! + (c*ρ)^c / (c!(1-ρ))]^(-1)
#   C_erlang = ((c*ρ)^c / c!) * (1/(1-ρ)) * P0
#   Wq = C_erlang / (c * μ * (1-ρ))

pass

In [ ]:
# === Exercise 8: Solution ===

lam = 20.0
mu = 0.5

print("=" * 70)
print("EXERCISE 8: QUEUE STABILITY")
print("=" * 70)
print(f"\nArrival rate λ = {lam} req/s")
print(f"Service rate μ = {mu} req/s per GPU (avg service time = {1/mu:.0f}s)")

# Part 1: Minimum GPUs for stability
offered_load = lam / mu  # = 40
c_min_stable = ceil(offered_load)  # need c > λ/μ, so c ≥ 41

print(f"\nPart 1: Minimum GPUs for stability")
print(f"  Offered load a = λ/μ = {lam}/{mu} = {offered_load:.0f}")
print(f"  Need c > {offered_load:.0f} → c_min = {c_min_stable}")
rho_min = lam / (c_min_stable * mu)
print(f"  ρ at c={c_min_stable}: {rho_min:.4f}")

# Part 2: GPUs for ρ < 0.7
c_target = ceil(lam / (mu * 0.7))
rho_target = lam / (c_target * mu)

print(f"\nPart 2: GPUs for ρ < 0.7")
print(f"  c ≥ λ / (μ × 0.7) = {lam} / ({mu} × 0.7) = {lam/(mu*0.7):.1f}")
print(f"  c = {c_target}")
print(f"  ρ = {lam} / ({c_target} × {mu}) = {rho_target:.4f}")

# Part 3: M/M/c wait time
def mmc_wq(lam, mu, c):
    """Compute M/M/c mean waiting time."""
    rho = lam / (c * mu)
    a = lam / mu  # offered load
    # P0
    sum_terms = sum(a**k / factorial(k) for k in range(c))
    last_term = a**c / (factorial(c) * (1 - rho))
    P0 = 1.0 / (sum_terms + last_term)
    # Erlang-C
    C = (a**c / factorial(c)) * (1 / (1 - rho)) * P0
    # Wq
    Wq = C / (c * mu * (1 - rho))
    return Wq, rho, C, P0

Wq, rho, C_erlang, P0 = mmc_wq(lam, mu, c_target)

print(f"\nPart 3: M/M/c analysis with c = {c_target}")
print(f"  ρ = {rho:.4f}")
print(f"  P0 = {P0:.6f}")
print(f"  P(queue) = {C_erlang:.4f}")
print(f"  E[Wq] = {Wq:.4f} s = {Wq*1000:.2f} ms")
print(f"  E[W] (total time in system) = {Wq + 1/mu:.4f} s")

# Compare with other GPU counts
print(f"\n  Comparison across GPU counts:")
headers = ["GPUs (c)", "ρ", "Wq (ms)", "P(queue)"]
rows = []
for c in [c_min_stable, 45, 50, c_target, 65, 70, 80]:
    try:
        w, r, ce, _ = mmc_wq(lam, mu, c)
        if r < 1:
            rows.append([str(c), f"{r:.3f}", f"{w*1000:.1f}", f"{ce:.4f}"])
        else:
            rows.append([str(c), f"{r:.3f}", "∞", "—"])
    except:
        rows.append([str(c), "—", "—", "—"])
print_table(headers, rows, col_width=14)

print("\n" + "=" * 70)
print("  ALL EXERCISES COMPLETE")
print("=" * 70)
print()
print("  Summary of key results:")
print("  Ex1: TPOT constant at ~4.8ms until B > 295 (BW→compute transition)")
print("  Ex2: 70B BF16 TP=2 → ~137 concurrent 2048-token requests")
print("  Ex3: Continuous batching ≈ s_max/E[s] ≈ 2–5× over static")
print("  Ex4: PagedAttention wastes ~200× less memory")
print("  Ex5: Spec decode 2.18× speedup at α=0.75, K=4")
print("  Ex6: ~1 prefill GPU per 19 decode GPUs (500/200 in/out)")
print("  Ex7: INT4 saves ~$3K/month for 1B tokens/day")
print("  Ex8: 58+ GPUs for ρ<0.7 with λ=20 req/s, μ=0.5")